In [2]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# 1. Load Data
df_laps = spark.read.table("silver_all_laps")
df_weather = spark.read.table("silver_all_weather")

# 2. CONVERT ALL LAP TIMES TO SECONDS (Do this first so it applies to both FP2 and Q)
df_laps = df_laps.withColumn("Hours", F.regexp_extract("LapTime", r"(\d{2}):(\d{2}):(\d{2}\.\d+)", 1).cast("double")) \
                 .withColumn("Minutes", F.regexp_extract("LapTime", r"(\d{2}):(\d{2}):(\d{2}\.\d+)", 2).cast("double")) \
                 .withColumn("Seconds", F.regexp_extract("LapTime", r"(\d{2}):(\d{2}):(\d{2}\.\d+)", 3).cast("double")) \
                 .withColumn("LapTimeSeconds", (F.col("Hours") * 3600) + (F.col("Minutes") * 60) + F.col("Seconds"))

# ====================================================================
# PART A: THE FRIDAY RACE SIMULATION (FP2)
# ====================================================================
valid_tires = ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']
df_fp2 = df_laps.filter((F.col("Session") == "fp2") & (F.col("TireCompound").isin(valid_tires)) & (F.col("IsAccurate") == "True"))

# Group by Driver to get their Average Race Pace
df_fp2_pace = df_fp2.groupBy("Year", "Track", "DriverNumber", "Team").agg(
    F.avg("LapTimeSeconds").alias("Average_Long_Run_Pace")
)

# Calculate the Delta to the fastest FP2 car
session_window = Window.partitionBy("Year", "Track")
df_fp2_deltas = df_fp2_pace.withColumn("Session_Fastest_Avg", F.min("Average_Long_Run_Pace").over(session_window)) \
                           .withColumn("Pace_Delta_To_Leader", F.col("Average_Long_Run_Pace") - F.col("Session_Fastest_Avg"))


# ====================================================================
# PART B: THE SATURDAY QUALIFYING SHOOTOUT (Q)
# ====================================================================
# Filter for Q session. We just want their absolute fastest lap of the session.
df_q = df_laps.filter(F.col("Session") == "q")

df_q_pace = df_q.groupBy("Year", "Track", "DriverNumber").agg(
    F.min("LapTimeSeconds").alias("Qualifying_Lap_Pace")
)

# Calculate the Delta to Pole Position and their Simulated Grid Rank
df_q_deltas = df_q_pace.withColumn("Pole_Lap_Time", F.min("Qualifying_Lap_Pace").over(session_window)) \
                       .withColumn("Qualifying_Delta_To_Pole", F.col("Qualifying_Lap_Pace") - F.col("Pole_Lap_Time"))

grid_window = Window.partitionBy("Year", "Track").orderBy("Qualifying_Delta_To_Pole")
df_q_deltas = df_q_deltas.withColumn("Simulated_Grid_Position", F.rank().over(grid_window))


# ====================================================================
# PART C: THE SATURDAY NIGHT MASTER JOIN
# ====================================================================
# Join FP2 Race Pace with Q Grid Position! (Inner join ensures we only keep drivers who participated in both)
df_master = df_fp2_deltas.join(df_q_deltas, on=["Year", "Track", "DriverNumber"], how="inner")

# Add the Weather (Using FP2 weather as our proxy for weekend track temps)
df_fp2_weather = df_weather.filter(F.col("Session") == "fp2").groupBy("Year", "Track").agg(F.round(F.avg("TrackTemp"), 1).alias("AvgTrackTemp"))
df_final = df_master.join(df_fp2_weather, on=["Year", "Track"], how="left")

# Save the ultimate Saturday Night Predictor Table
df_final.write.format("delta").mode("overwrite").saveAsTable("gold_saturday_predictor_features")

print("✅ Success! You have forged the Saturday Night Predictor.")

# Let's check Australia 2026 to see their Qualifying Deltas!
display(df_final.select("Year", "Track", "DriverNumber", "Team", "Simulated_Grid_Position", "Qualifying_Delta_To_Pole", "Pace_Delta_To_Leader")
        .filter((F.col("Track") == "australia") & (F.col("Year") == "2026"))
        .orderBy("Simulated_Grid_Position"))

StatementMeta(, 5c38c440-0388-417d-bde9-a7a11da97d47, 4, Finished, Available, Finished, False)

✅ Success! You have forged the Saturday Night Predictor.


SynapseWidget(Synapse.DataFrame, 2337a3d6-2ff1-4318-b1cd-1e01825d6a1d)